# Station Stacking v10 - KDAL

Experimental notebook for `KDAL`.

This version keeps the v9 live-safe GFS/HRRR/NBM remaining-warmup data contract, trains only CatBoost with tuned Huber loss, disables ridge stacking, and writes artifacts to `data/calibration/station_stacking_v10`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 0
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 0
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v10_catboost_huber"
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V10_DROPPED_FEATURE_COLUMNS,
    V10_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V10 Contract

`feature_version="v10"` keeps the v9 feature contract, uses `target_mode="remaining_warmup"`, and restricts the experiment to CatBoost with tuned Huber loss. Ridge stacking is disabled so the final exported model is the CatBoost base learner.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V10_FEATURE_COLUMNS, sorted(V10_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
6,KDAL,gfs,1983,2021-01-01,2026-06-10
7,KDAL,hrrr,1987,2021-01-01,2026-06-10
8,KDAL,nbm,1986,2021-01-01,2026-06-10


## Model Scores


In [6]:
missing_packages = missing_model_dependencies(("catboost",))
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v10",
    target_mode="remaining_warmup",
    base_model_methods=("catboost",),
    stack_enabled=False,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v10/KDAL_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-16 12:58:16,537] A new study created in RDB with name: KDAL_v10_remaining_warmup_base_catboost_mae_f
[W 2026-06-16 13:12:05,369] Trial 0 failed with parameters: {'iterations': 2931, 'learning_rate': 0.001783588943232587, 'depth': 10, 'l2_leaf_reg': 0.9942958882300413, 'random_strength': 7.186216756161439, 'bagging_temperature': 12.184767612363034, 'border_count': 110, 'rsm': 0.6158971964095841, 'huber_delta': 1.5} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "d:\dev\weather-research\.venv\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "D:\dev\weather-research\src\calibration\station_stacking.py", line 1153, in objective
    predicted, fit_metadata = _fit_predict_base_model(
                              ~~~~~~~~~~~~~~~~~~~~~~~^
        config=config,
        ^^^^^^^^^^^^^^
    ...<6 lines>...
        early_stopping=True,
        ^^^^^^^^^^^^^^^^^^^^
    )
    ^

KeyboardInterrupt: 

In [ ]:
exported_weights = export_station_model_weights(
    project_root=PROJECT_ROOT,
    station_id=STATION_ID,
    artifact_dir=config.resolved_output_dir(),
    model_version=MODEL_VERSION,
    timing_mode=config.timing_mode,
    providers=tuple(config.providers),
    feature_version=config.effective_feature_version,
    optuna_metric=config.effective_optuna_metric,
    target_mode=config.effective_target_mode,
    base_model_methods=tuple(config.effective_base_model_methods),
    stack_enabled=config.stack_enabled,
    source_pipeline="notebooks/experiments/station_stacking_v10",
)

exported_weights.bundle_path, exported_weights.manifest_path


## V10 Feature Coverage


In [ ]:
v10_feature_coverage = (
    result.features[V10_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v10_feature_coverage


In [ ]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V10_FEATURE_COLUMNS)]


## Dropped Feature Check


In [ ]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V10_DROPPED_FEATURE_COLUMNS)
]

dropped_present


## Morning Trend Coverage


In [ ]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


## Rounded Within 1F Accuracy


In [ ]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


## Version Comparison


In [ ]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


## 2026 OOF Weather Brackets


In [ ]:
result.bracket_metrics
